# 05. AI 3줄 요약 (GPT-4o-mini)

이 Notebook에서 하는 일:
1. 포트폴리오 현황 데이터 수집 (자산, 위험점수, 인출 정보)
2. GPT-4o-mini로 한국어 3줄 요약 생성
3. API 실패 시 룰 기반 폴백(fallback) 자동 적용
4. 요약 결과 DB 저장
5. 과거 요약 이력 조회

In [1]:
# 공통 설정
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date
from openai import OpenAI

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
WITHDRAWAL_CFG  = CONFIG['withdrawal']
USER_NAME       = CONFIG['user']['name']
inflation_rate  = CONFIG.get('inflation', {}).get('assumed_rate', 0.025)

# 국민연금 설정
PENSION_CFG    = CONFIG.get('income', {}).get('national_pension', {})
base_pension   = PENSION_CFG.get('base_amount', 0)
pension_start  = PENSION_CFG.get('start_date', None)

today_date = date.today()
if pension_start:
    py, pm = map(int, pension_start.split('-'))
    if today_date >= date(py, pm, 1):
        years_since = (today_date.year - py) + (today_date.month - pm) / 12
        pension_income = base_pension * (1 + inflation_rate) ** years_since
        months_to_pension = 0
    else:
        pension_income = 0
        months_to_pension = (py - today_date.year) * 12 + (pm - today_date.month)
else:
    pension_income = 0
    months_to_pension = 0

api_key = os.getenv('OPENAI_API_KEY', '')
client  = OpenAI(api_key=api_key) if api_key else None

print(f"사용자: {USER_NAME}")
print(f"OpenAI API: {'연결됨' if client else '키 없음 — 룰 기반 폴백 사용'}")
if pension_income > 0:
    print(f"국민연금 수령 중: {pension_income:,.0f}원/월")
else:
    print(f"국민연금 개시까지: {months_to_pension}개월 ({pension_start}, 월 {base_pension:,.0f}원 예정)")

사용자: 종헌
OpenAI API: 연결됨
국민연금 개시까지: 59개월 (2031-04, 월 2,245,452원 예정)


In [2]:
# 자산 데이터 로드
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'

if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig')
    print('※ 샘플 데이터 사용 중')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)


# ── 비활성(만료) 자산 제외 ──────────────────────────────────
if 'is_active' not in df.columns: df['is_active'] = 1
if 'maturity_date' not in df.columns: df['maturity_date'] = ''
df['is_active'] = df['is_active'].fillna(1).astype(int)
inactive_n = int((df['is_active'] == 0).sum())
df = df[df['is_active'] == 1].reset_index(drop=True)
if inactive_n > 0:
    print(f'ℹ️  비활성(만료) 자산 {inactive_n}개 제외 → 활성 {len(df)}개 사용')
BUCKET_MAP = {'cash': 1, 'bond': 2, 'tdf': 2, 'fund': 2, 'equity': 3, 'income': 3}
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
type_sum   = df.groupby('asset_type')['current_value'].sum()
total      = df['current_value'].sum()
b1 = bucket_sum.get(1, 0)
b2 = bucket_sum.get(2, 0)
b3 = bucket_sum.get(3, 0)

print(f'총 자산: {total:,.0f}원  |  버킷1: {b1:,.0f}  버킷2: {b2:,.0f}  버킷3: {b3:,.0f}')


총 자산: 804,317,346원  |  버킷1: 334,841,532  버킷2: 161,495,074  버킷3: 307,980,740


In [3]:
# DB에서 최신 위험 점수 + 인출 데이터 로드
db_path = PROJECT_ROOT / 'portfolio.db'
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()

# 위험 점수
cur.execute('SELECT total_score, cash_score, seq_score, conc_score, level FROM risk_scores ORDER BY date DESC LIMIT 1')
risk_row = cur.fetchone()

# 인출 데이터 — actual_amount 우선, 없으면 amount(권장) 사용
cur.execute('''
    SELECT amount, actual_amount, guardrail_applied, note
    FROM withdrawal_log
    ORDER BY date DESC LIMIT 1
''')
wd_row = cur.fetchone()
conn.close()

if risk_row:
    total_score, cash_score, seq_score, conc_score, risk_level = risk_row
else:
    total_score, cash_score, seq_score, conc_score, risk_level = 0, 0, 0, 0, 'green'
    print('※ 위험 점수 없음 — 03_risk_score.ipynb를 먼저 실행하세요')

if wd_row:
    recommended_withdrawal = wd_row[0] or 0
    actual_withdrawal      = wd_row[1]          # None이면 미입력
    guardrail_applied      = wd_row[2] or 0
    wd_note                = wd_row[3] or ''

    # 실제 입력값 우선, 없으면 권장값 사용
    if actual_withdrawal is not None:
        display_withdrawal = actual_withdrawal
        withdrawal_label   = '실제 인출액'
    else:
        display_withdrawal = recommended_withdrawal
        withdrawal_label   = '권장 인출액 (실제 미입력)'
else:
    recommended_withdrawal = MONTHLY_EXPENSE - pension_income
    actual_withdrawal      = None
    display_withdrawal     = recommended_withdrawal
    withdrawal_label       = '기본값 (02_bucket_engine 실행 필요)'
    guardrail_applied      = 0
    wd_note                = ''

months_covered = b1 / MONTHLY_EXPENSE if MONTHLY_EXPENSE > 0 else 0
equity_ratio   = (type_sum.get('equity', 0) + type_sum.get('income', 0)) / total if total > 0 else 0
cash_ratio     = type_sum.get('cash', 0) / total if total > 0 else 0
bond_ratio     = (type_sum.get('bond', 0) + type_sum.get('tdf', 0) + type_sum.get('fund', 0)) / total if total > 0 else 0

level_map = {'green': '녹색(안전)', 'yellow': '황색(주의)', 'red': '적색(위험)'}
print(f'위험 등급: {level_map.get(risk_level, risk_level)}  |  종합 점수: {total_score:.1f}점')
print(f'현금 {months_covered:.1f}개월치  |  주식 비중 {equity_ratio*100:.1f}%')
print(f'{withdrawal_label}: {display_withdrawal:,.0f}원')

위험 등급: 황색(주의)  |  종합 점수: 30.0점
현금 67.0개월치  |  주식 비중 38.3%
실제 인출액: 4,500,000원


## 1. 룰 기반 폴백(Fallback) 요약 생성

In [4]:
def rule_based_summary(total, b1, months_covered, equity_ratio, total_score,
                       risk_level, display_withdrawal, monthly_expense,
                       guardrail_applied, withdrawal_label,
                       pension_income, months_to_pension, base_pension):

    level_txt = {'green': '안전', 'yellow': '주의', 'red': '위험'}.get(risk_level, '확인 필요')

    # 줄 1: 전체 현황
    line1 = (f"총 자산 {total/1e8:.2f}억원, 위험 등급 {level_txt}({total_score:.0f}점)으로 "
             f"{'포트폴리오는 안정적입니다.' if risk_level == 'green' else '점검이 필요합니다.'}")

    # 줄 2: 현금성 자산
    if months_covered >= 12:
        line2 = f"현금성 자산은 {months_covered:.0f}개월치({b1/1e8:.2f}억원)로 충분히 확보되어 있습니다."
    elif months_covered >= 6:
        line2 = f"현금성 자산이 {months_covered:.0f}개월치({b1/1e8:.2f}억원)로 12개월치 확보를 권장합니다."
    else:
        line2 = f"현금성 자산이 {months_covered:.0f}개월치({b1/1e8:.2f}억원)로 부족합니다. 즉시 보충을 권장합니다."

    # 줄 3: 인출 및 연금
    guard_txt = ' (가드레일 하향 적용)' if guardrail_applied else ''
    is_actual = '실제' in withdrawal_label
    label_short = '실제 인출액' if is_actual else '권장 인출액'

    if pension_income > 0:
        line3 = (f"이번 달 {label_short}은 {display_withdrawal:,.0f}원{guard_txt}이며, "
                 f"국민연금 {pension_income:,.0f}원과 합산 시 생활비가 충당됩니다.")
    elif months_to_pension > 0:
        line3 = (f"이번 달 {label_short}은 {display_withdrawal:,.0f}원{guard_txt}이며, "
                 f"{months_to_pension}개월 후 국민연금({base_pension:,.0f}원) 개시 시 인출 부담이 줄어듭니다.")
    else:
        line3 = (f"이번 달 {label_short}은 {display_withdrawal:,.0f}원{guard_txt}입니다.")

    return f"{line1}\n{line2}\n{line3}"


fallback_summary = rule_based_summary(
    total, b1, months_covered, equity_ratio, total_score,
    risk_level, display_withdrawal, MONTHLY_EXPENSE,
    guardrail_applied, withdrawal_label,
    pension_income, months_to_pension, base_pension
)

print('=== 룰 기반 폴백 요약 (미리보기) ===')
print(fallback_summary)

=== 룰 기반 폴백 요약 (미리보기) ===
총 자산 8.04억원, 위험 등급 주의(30점)으로 점검이 필요합니다.
현금성 자산은 67개월치(3.35억원)로 충분히 확보되어 있습니다.
이번 달 실제 인출액은 4,500,000원이며, 59개월 후 국민연금(2,245,452원) 개시 시 인출 부담이 줄어듭니다.


## 2. GPT-4o-mini AI 요약 생성

In [5]:
def build_prompt(total, b1, b2, b3, months_covered, equity_ratio, cash_ratio, bond_ratio,
                 total_score, cash_score, seq_score, conc_score, risk_level,
                 display_withdrawal, monthly_expense, guardrail_applied, withdrawal_label,
                 pension_income, months_to_pension, base_pension, user_name):

    level_txt    = {'green': '녹색(안전)', 'yellow': '황색(주의)', 'red': '적색(위험)'}.get(risk_level, risk_level)
    is_actual    = '실제' in withdrawal_label
    net_from_portfolio = monthly_expense - pension_income

    pension_section = ''
    if pension_income > 0:
        pension_section = f'- 국민연금 수령 중: {pension_income:,.0f}원/월\n'
        pension_section += f'- 포트폴리오 실 인출 (생활비-연금): {net_from_portfolio:,.0f}원/월\n'
    elif months_to_pension > 0:
        pension_section = f'- 국민연금 미수령 ({months_to_pension}개월 후 개시 예정: {base_pension:,.0f}원/월)\n'
        pension_section += f'- 포트폴리오에서 생활비 전액 인출 중\n'

    prompt = f"""당신은 은퇴자 {user_name}님의 포트폴리오를 관리하는 AI 자문가입니다.
아래 데이터를 바탕으로 이번 달 포트폴리오 현황을 친근하고 명확한 한국어로 정확히 3줄로 요약해 주세요.

**포트폴리오 현황 ({date.today().strftime('%Y년 %m월')})**
- 총 자산: {total/1e8:.2f}억원
- 버킷 1 (현금성): {b1/1e8:.2f}억원 ({months_covered:.1f}개월치 생활비)
- 버킷 2 (채권/TDF/펀드): {b2/1e8:.2f}억원
- 버킷 3 (성장/인컴): {b3/1e8:.2f}억원
- 자산 비중: 현금 {cash_ratio*100:.1f}% / 채권 {bond_ratio*100:.1f}% / 주식형 {equity_ratio*100:.1f}%

**위험 점수**
- 종합 위험 점수: {total_score:.1f}점 / 100점  등급: {level_txt}
- 현금 위험: {cash_score:.0f}점  순서 위험: {seq_score:.0f}점  집중 위험: {conc_score:.0f}점

**수입·지출 구조**
- 월 생활비: {monthly_expense:,.0f}원
{pension_section}- {withdrawal_label}: {display_withdrawal:,.0f}원
- 가드레일 적용: {'예 (인출 하향 조정)' if guardrail_applied else '아니오 (정상)'}

**요약 규칙**
1. 정확히 3줄, 각 줄은 완결된 문장
2. 첫 줄: 전체 현황 및 위험 등급
3. 둘째 줄: 현금성 자산 충분성 및 필요 조치
4. 셋째 줄: 이번 달 {'실제' if is_actual else '권장'} 인출액 및 국민연금 관련 사항
5. 숫자는 억원 단위로 표기, 전문 용어는 쉽게 설명
6. 위험 등급이 황색/적색이면 구체적 행동 권고 포함"""

    return prompt


def generate_ai_summary():
    if not client:
        print('API 키 없음 → 룰 기반 폴백 사용')
        return fallback_summary, 'fallback'

    prompt = build_prompt(
        total, b1, b2, b3, months_covered, equity_ratio, cash_ratio, bond_ratio,
        total_score, cash_score, seq_score, conc_score, risk_level,
        display_withdrawal, MONTHLY_EXPENSE, guardrail_applied, withdrawal_label,
        pension_income, months_to_pension, base_pension, USER_NAME
    )

    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': '당신은 은퇴 자산관리 전문 AI입니다. 항상 한국어로 답변합니다.'},
                {'role': 'user', 'content': prompt}
            ],
            max_tokens=300,
            temperature=0.4
        )
        summary = response.choices[0].message.content.strip()
        return summary, 'gpt-4o-mini'
    except Exception as e:
        print(f'API 오류: {e}')
        print('→ 룰 기반 폴백 자동 적용')
        return fallback_summary, 'fallback'


print('GPT-4o-mini 요약 생성 중...')
summary, source = generate_ai_summary()

print()
print(f'=== AI 포트폴리오 요약 ({date.today().strftime("%Y년 %m월 %d일")}) ===')
print(f'생성 방식: {source}')
print(f'인출액 기준: {withdrawal_label}')
print()
print(summary)

GPT-4o-mini 요약 생성 중...

=== AI 포트폴리오 요약 (2026년 05월 21일) ===
생성 방식: gpt-4o-mini
인출액 기준: 실제 인출액

2026년 5월 현재 포트폴리오는 총 8.04억원이며, 종합 위험 점수는 30.0점으로 황색(주의) 등급입니다. 현금성 자산 3.35억원은 67개월치 생활비로 충분하지만, 위험 점수를 낮추기 위해 자산 배분을 재조정하는 것이 필요합니다. 이번 달 실제 인출액은 4.5백만원이며, 국민연금은 59개월 후에 개시되어 월 224만원을 받을 예정입니다.


## 3. 요약 DB 저장

In [6]:
today   = str(date.today())
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()

cur.execute("SELECT id FROM recommendations WHERE date=? AND rule_id='ai_summary'", (today,))
existing = cur.fetchone()

if existing:
    cur.execute("""
        UPDATE recommendations SET message=?, status='completed'
        WHERE date=? AND rule_id='ai_summary'
    """, (summary, today))
    print(f'오늘({today}) AI 요약 업데이트 완료')
else:
    cur.execute("""
        INSERT INTO recommendations (date, rule_id, message, status)
        VALUES (?, 'ai_summary', ?, 'completed')
    """, (today, summary))
    print(f'오늘({today}) AI 요약 저장 완료')

conn.commit()
conn.close()
print(f'생성 방식: {source}  |  인출액 기준: {withdrawal_label}')

오늘(2026-05-21) AI 요약 업데이트 완료
생성 방식: gpt-4o-mini  |  인출액 기준: 실제 인출액


## 4. 과거 요약 이력 조회

In [7]:
conn = sqlite3.connect(db_path)
history = pd.read_sql_query("""
    SELECT date, message FROM recommendations
    WHERE rule_id = 'ai_summary'
    ORDER BY date DESC LIMIT 6
""", conn)
conn.close()

if history.empty:
    print('저장된 AI 요약이 없습니다.')
else:
    print('=== 최근 AI 요약 이력 ===')
    print()
    for _, row in history.iterrows():
        print(f'[{row["date"]}]')
        print(row['message'])
        print()

=== 최근 AI 요약 이력 ===

[2026-05-21]
2026년 5월 현재 포트폴리오는 총 8.04억원이며, 종합 위험 점수는 30.0점으로 황색(주의) 등급입니다. 현금성 자산 3.35억원은 67개월치 생활비로 충분하지만, 위험 점수를 낮추기 위해 자산 배분을 재조정하는 것이 필요합니다. 이번 달 실제 인출액은 4.5백만원이며, 국민연금은 59개월 후에 개시되어 월 224만원을 받을 예정입니다.

[2026-05-20]
현재 종헌님의 포트폴리오는 총 8.91억원이며, 위험 등급은 황색(주의)입니다. 현금성 자산이 4.22억원으로 충분하지만, 집중 위험이 높아 자산 분산을 고려해야 합니다. 이번 달 권장 인출액은 297만원이며, 생활비를 고려해 인출 시 주의가 필요합니다.



---

## 완료!

- 실제 인출액(11_withdrawal_input에서 입력)이 있으면 그 값으로 요약합니다.
- 미입력 시 시스템 권장액으로 대체합니다.

**다음 단계**: `06_dashboard.ipynb` — 통합 대시보드